# Inventario técnico de los datasets de Airbnb

## Objetivo

Examinar la estructura de los seis archivos CSV originales antes de realizar
cualquier limpieza, transformación o análisis exploratorio.

## Preguntas iniciales

- ¿Están disponibles los seis archivos esperados?
- ¿Cuántas filas y columnas tiene cada ciudad?
- ¿Comparten las mismas columnas?
- ¿Qué tipos de datos infiere Pandas?
- ¿Qué diferencias de esquema deberán resolverse posteriormente?

Los archivos originales se leerán sin modificarlos.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_directory = project_root / "data" / "raw" / "airbnb"
csv_files = sorted(data_directory.glob("*.csv"))

assert len(csv_files) == 6, (
    f"Se esperaban 6 archivos CSV, pero se encontraron {len(csv_files)}"
)

[(csv_file.name, csv_file.stat().st_size) for csv_file in csv_files]

[('london_airbnb.csv', 11578155),
 ('madrid_airbnb.csv', 2801783),
 ('milan_airbnb.csv', 2393568),
 ('NY_airbnb.csv', 7077973),
 ('sydney_airbnb.csv', 5504518),
 ('tokyo_airbnb.csv', 1738173)]

## 1. Carga controlada de los archivos

Se cargan los seis CSV completos en memoria. Pandas infiere inicialmente los
tipos de datos, pero en esta etapa no se limpia ni transforma ninguna columna.

In [5]:
datasets = {}

for csv_file in csv_files:
    datasets[csv_file.name] = pd.read_csv(
        csv_file,
        low_memory=False,
    )

len(datasets)

6

In [6]:
basic_inventory = pd.DataFrame(
    [
        {
            "file_name": file_name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
        }
        for file_name, dataframe in datasets.items()
    ]
).sort_values("file_name", ignore_index=True)

basic_inventory

,file_name,rows,columns
0,NY_airbnb.csv,48895,16
1,london_airbnb.csv,85068,16
2,madrid_airbnb.csv,19618,16
3,milan_airbnb.csv,18322,15
4,sydney_airbnb.csv,36662,16
5,tokyo_airbnb.csv,11466,14


## 2. Comparación de los esquemas

Se comparan los nombres de las columnas para determinar cuáles están presentes
en todas las ciudades y cuáles aparecen solamente en algunos datasets.

En esta etapa solo se observa la estructura: no se renombran, eliminan ni crean
columnas.

In [8]:
all_columns = sorted(
    {
        column
        for dataframe in datasets.values()
        for column in dataframe.columns
    }
)

schema_matrix = pd.DataFrame(
    {
        file_name: [
            column in dataframe.columns
            for column in all_columns
        ]
        for file_name, dataframe in datasets.items()
    },
    index=all_columns,
)

schema_matrix.index.name = "column"
schema_matrix

,london_airbnb.csv,madrid_airbnb.csv,milan_airbnb.csv,NY_airbnb.csv,sydney_airbnb.csv,tokyo_airbnb.csv
column,,,,,,
availability_365,True,True,True,True,True,False
calculated_host_listings_count,True,True,True,True,True,False
host_id,True,True,True,True,True,True
host_name,True,True,True,True,True,True
id,True,True,True,True,True,True
last_review,True,True,True,True,True,True
latitude,True,True,True,True,True,True
longitude,True,True,True,True,True,True
minimum_nights,True,True,True,True,True,True


In [11]:
schema_differences = schema_matrix.loc[
    ~schema_matrix.all(axis=1)
]

schema_differences

,london_airbnb.csv,madrid_airbnb.csv,milan_airbnb.csv,NY_airbnb.csv,sydney_airbnb.csv,tokyo_airbnb.csv
column,,,,,,
availability_365,True,True,True,True,True,False
calculated_host_listings_count,True,True,True,True,True,False
neighbourhood_group,True,True,False,True,True,True


### Interpretación de las diferencias

Los seis datasets comparten 13 columnas. El conjunto completo contiene 16
columnas diferentes y se observaron tres diferencias de esquema:

- Tokio no contiene `availability_365`.
- Tokio no contiene `calculated_host_listings_count`.
- Milán no contiene `neighbourhood_group`.

Estas ausencias explican las diferencias en el número total de columnas. No se
consideran todavía errores de calidad, porque podrían responder a diferencias en
la información publicada para cada ciudad.

Antes de combinar los datasets será necesario decidir cómo representar estas
columnas ausentes. En este inventario no se realiza esa transformación.

In [13]:
dtype_matrix = pd.DataFrame(
    {
        file_name: dataframe.dtypes.astype(str)
        for file_name, dataframe in datasets.items()
    }
).reindex(all_columns)

dtype_matrix.index.name= "column"
dtype_matrix

,london_airbnb.csv,madrid_airbnb.csv,milan_airbnb.csv,NY_airbnb.csv,sydney_airbnb.csv,tokyo_airbnb.csv
column,,,,,,
availability_365,int64,int64,int64,int64,int64,NaN
calculated_host_listings_count,int64,int64,int64,int64,int64,NaN
host_id,int64,int64,int64,int64,int64,int64
host_name,str,str,str,str,str,str
id,int64,int64,int64,int64,int64,int64
last_review,str,str,str,str,str,str
latitude,float64,float64,float64,float64,float64,float64
longitude,float64,float64,float64,float64,float64,float64
minimum_nights,int64,int64,int64,int64,int64,int64


In [15]:
dtype_variation_count = dtype_matrix.nunique(
    axis=1,
    dropna=True,
)

dtype_differences = dtype_matrix.loc[
    dtype_variation_count > 1
]

dtype_differences

,london_airbnb.csv,madrid_airbnb.csv,milan_airbnb.csv,NY_airbnb.csv,sydney_airbnb.csv,tokyo_airbnb.csv
column,,,,,,
neighbourhood_group,float64,str,NaN,str,float64,float64
